In [1]:
import pandas as pd
import numpy as np
import torch
from torch.nn import CrossEntropyLoss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
    

2025-12-01 10:45:23.930674: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-01 10:45:23.930700: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-01 10:45:23.931954: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-01 10:45:23.937987: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Import dataset

In [2]:
COLS = [
    "json_id","label","statement","subjects","speaker",
    "speaker_job","state_info","party_affiliation",
    "barely_true_counts","false_counts","half_true_counts",
    "mostly_true_counts","pants_on_fire_counts",
    "context","justification"
]

def read_liarplus_tsv(path):
    df = pd.read_csv(path, sep="\t", header=None, names=COLS, quoting=3, on_bad_lines="skip")
    for c in ["barely_true_counts","false_counts","half_true_counts","mostly_true_counts","pants_on_fire_counts"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    df.dropna(subset=["speaker","statement"], inplace=True)
    return df

df = read_liarplus_tsv("train2.tsv")

In [3]:
speakers = df["speaker"].dropna().unique()
train_speakers, test_speakers = train_test_split(speakers, test_size=0.2, random_state=42)
train_df = df[df["speaker"].isin(train_speakers)].copy()
test_df  = df[df["speaker"].isin(test_speakers)].copy()

In [4]:
train_speakers = set(train_df["speaker"])
test_speakers  = set(test_df["speaker"])
len(train_speakers & test_speakers)


0

In [5]:
# Split the speakers between training and test speaker to avoid overlap

In [6]:
speakers = df["speaker"].dropna().unique()
train_speakers, test_speakers = train_test_split(speakers, test_size=0.2, random_state=42)
train_df = df[df["speaker"].isin(train_speakers)].copy()
test_df  = df[df["speaker"].isin(test_speakers)].copy()

print(f"Train speakers: {len(train_speakers)}, Test speakers: {len(test_speakers)}")
print("Overlap:", len(set(train_speakers) & set(test_speakers)))

Train speakers: 2332, Test speakers: 584
Overlap: 0


In [7]:
# Given a speaker, calculate their reputation score using history

In [8]:
def build_speaker_agg(df):
    hist_cols = [
        "barely_true_counts","false_counts","half_true_counts",
        "mostly_true_counts","pants_on_fire_counts"
    ]
    agg = df.groupby("speaker", as_index=False)[hist_cols].max()
    agg["num_statements"] = df.groupby("speaker")["statement"].count().values
    h = agg[hist_cols].astype(float)
    total = h.sum(axis=1).clip(lower=1.0)
    w = {"barely_true_counts":0.25,"half_true_counts":0.5,"mostly_true_counts":1.0,
         "false_counts":0.0,"pants_on_fire_counts":0.0}
    score = (
        w["barely_true_counts"]*h["barely_true_counts"] +
        w["half_true_counts"]*h["half_true_counts"] +
        w["mostly_true_counts"]*h["mostly_true_counts"]
    ) / total
    agg["rep_score"] = score
    bins = [-np.inf,0.3,0.6,np.inf]
    labels = ["low","medium","high"]
    agg["rep_label"] = pd.cut(score,bins=bins,labels=labels)
    return agg

speaker_reputation_train = build_speaker_agg(train_df)
speaker_reputation_test  = build_speaker_agg(test_df)

In [9]:
# Combine meta data with statement to make 1 big string to pass into model later

In [10]:
def merge_and_build_text(df, rep_table):
    df = df.merge(rep_table[["speaker","rep_label","num_statements"]], on="speaker", how="left")
    df.dropna(subset=["rep_label"], inplace=True)
    for c in ["party_affiliation","state_info","speaker_job"]:
        df[c] = df[c].fillna("Unknown")
    df["input_text"] = (
        "Party: " + df["party_affiliation"].astype(str)
        + " | State: " + df["state_info"].astype(str)
        + " | Job: " + df["speaker_job"].astype(str)
        + " | Statement: " + df["statement"].astype(str)
    )
    return df

train = merge_and_build_text(train_df, speaker_reputation_train)
test  = merge_and_build_text(test_df,  speaker_reputation_test)

In [11]:
# Encode labels and tokenize
le = LabelEncoder()
train["rep_label_id"] = le.fit_transform(train["rep_label"])
test["rep_label_id"]  = le.transform(test["rep_label"])

train["sample_weight"] = train["num_statements"].apply(
    lambda x: min(1.0, np.log1p(x) / np.log1p(train["num_statements"].max()))
)

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
train_enc = tokenizer(list(train["input_text"]), truncation=True, padding=True, max_length=128)
test_enc  = tokenizer(list(test["input_text"]), truncation=True, padding=True, max_length=128)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

/opt/conda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [12]:
class WeightedDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels, weights):
        self.enc = enc
        self.labels = labels
        self.weights = weights
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key,val in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        item["weights"] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item
    def __len__(self): return len(self.labels)

train_ds = WeightedDataset(train_enc, train["rep_label_id"].values, train["sample_weight"].values)
test_ds  = WeightedDataset(test_enc, test["rep_label_id"].values, np.ones(len(test)))

# Balance the weight
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train["rep_label_id"]),
    y=train["rep_label_id"]
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        weights = inputs.pop("weights")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device), reduction="none")
        losses = loss_fct(logits, labels)
        weighted_loss = torch.mean(losses * weights.to(losses.device))
        return (weighted_loss, outputs) if return_outputs else weighted_loss

In [13]:
#training the data
training_args = TrainingArguments(
    output_dir="./results_distil_meta",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    learning_rate=2.5e-5,
    weight_decay=0.01,
    fp16=True,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    remove_unused_columns=False,
    save_total_limit=1,
)

# model metrics to determine performance
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }


model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(le.classes_)
)

# Add dropout regularization to avoid overfitting
model.config.dropout = 0.2
model.config.attention_dropout = 0.2

# train the model
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
results = trainer.evaluate()
print("\nFinal Eval:", results)


/opt/conda/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.223600,1.521269,0.556593,0.471541
2,0.151700,1.781507,0.484615,0.438941
3,0.112900,2.380810,0.529121,0.442644



Final Eval: {'eval_loss': 1.5212689638137817, 'eval_accuracy': 0.5565934065934066, 'eval_macro_f1': 0.4715405987817854, 'eval_runtime': 1.2913, 'eval_samples_per_second': 1409.41, 'eval_steps_per_second': 176.564, 'epoch': 3.0}


In [14]:
preds = trainer.predict(test_ds)
y_true = preds.label_ids
y_pred = preds.predictions.argmax(axis=-1)
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=le.classes_))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))



Classification Report:
               precision    recall  f1-score   support

        high       0.39      0.14      0.21       322
         low       0.49      0.76      0.60       618
      medium       0.67      0.57      0.61       880

    accuracy                           0.56      1820
   macro avg       0.52      0.49      0.47      1820
weighted avg       0.56      0.56      0.54      1820


Confusion Matrix:
 [[ 45 159 118]
 [ 18 469 131]
 [ 51 330 499]]


## This model is trained on all unique speakers so that the neural network model learns pattern to identify reputation on unseen speakers. The next step is to save this model and build onto to for cases where there is overlapped speaker.

In [16]:
# use the last model
base_path = "./results_distil_meta/checkpoint-1057"  
model = DistilBertForSequenceClassification.from_pretrained(base_path)
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

/opt/conda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [17]:
# combine all the speakers (add all the speaker back to training dataset)
full_df = pd.concat([train_df, test_df]).drop_duplicates(subset=["statement"]).reset_index(drop=True)

# Rebuild speaker reputation and metadata-enhanced text
speaker_reputation_full = build_speaker_agg(full_df)
full = merge_and_build_text(full_df, speaker_reputation_full)

In [18]:
#tokenize the label
le = LabelEncoder()
full["rep_label_id"] = le.fit_transform(full["rep_label"])

full["sample_weight"] = full["num_statements"].apply(
    lambda x: min(1.0, np.log1p(x) / np.log1p(full["num_statements"].max()))
)

# Tokenize
full_enc = tokenizer(list(full["input_text"]), truncation=True, padding=True, max_length=128)

In [19]:
#weigh the class to handle data imbalance
full_ds = WeightedDataset(full_enc, full["rep_label_id"].values, full["sample_weight"].values)

In [20]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(full["rep_label_id"]),
    y=full["rep_label_id"]
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        weights = inputs.pop("weights")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device), reduction="none")
        losses = loss_fct(logits, labels)
        weighted_loss = torch.mean(losses * weights.to(losses.device))
        return (weighted_loss, outputs) if return_outputs else weighted_loss


In [27]:
# Define training configuration for HuggingFace Trainer
training_args = TrainingArguments(
    output_dir="./results_practical_combined",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=1e-5,
    fp16=True,
    weight_decay=0.01,
    logging_dir=None,
    save_total_limit=1,
    remove_unused_columns=False,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=full_ds)

trainer.train()


Step,Training Loss
500,0.197800
1000,0.184600
1500,0.168300
2000,0.157600
2500,0.150300
3000,0.137600
3500,0.143800


TrainOutput(global_step=3846, training_loss=0.16006910807132968, metrics={'train_runtime': 139.2798, 'train_samples_per_second': 220.822, 'train_steps_per_second': 27.613, 'total_flos': 891239993449344.0, 'train_loss': 0.16006910807132968, 'epoch': 3.0})

In [28]:
#save models
model.save_pretrained("./saved_reputation_model_practical")
tokenizer.save_pretrained("./saved_reputation_model_practical")


('./saved_reputation_model_practical/tokenizer_config.json',
 './saved_reputation_model_practical/special_tokens_map.json',
 './saved_reputation_model_practical/vocab.txt',
 './saved_reputation_model_practical/added_tokens.json',
 './saved_reputation_model_practical/tokenizer.json')

In [29]:
val = pd.read_csv('val2.tsv', sep ='\t', header=None)
val.columns = [
        "statement_id","json_id", "label","statement","subjects","speaker","speaker_job",
        "state_info","party_affiliation",
        "barely_true_counts","false_counts","half_true_counts",
        "mostly_true_counts","pants_on_fire_counts",
        "context","justification"
    ]

In [30]:
speaker_reputation_val = build_speaker_agg(val)
val_df = merge_and_build_text(val, speaker_reputation_val)

In [31]:
# Example for preparing validation dataset
val_df = val_df.copy().dropna(subset=["statement", "rep_label"])

# Convert labels to IDs (use the same LabelEncoder as training)
val_df["rep_label_id"] = le.transform(val_df["rep_label"])


In [32]:
tokenizer = DistilBertTokenizerFast.from_pretrained("./saved_reputation_model_practical")
model = DistilBertForSequenceClassification.from_pretrained("./saved_reputation_model_practical")
model.eval()

texts = list(val_df["input_text"])
labels = list(val_df["rep_label_id"])

preds = []
for i in range(0, len(texts), 8):  # process in batches
    batch = tokenizer(texts[i:i+8], padding=True, truncation=True, max_length=128, return_tensors="pt")
    with torch.no_grad():
        logits = model(**batch).logits
    preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())


In [33]:
acc = accuracy_score(labels, preds)
f1_macro = f1_score(labels, preds, average="macro")

print("Validation Accuracy:", round(acc, 3))
print("Validation Macro F1:", round(f1_macro, 3))

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=["low", "medium", "high"]))

print("Confusion Matrix:")
print(confusion_matrix(labels, preds))


Validation Accuracy: 0.752
Validation Macro F1: 0.693

Classification Report:
              precision    recall  f1-score   support

         low       0.50      0.60      0.55       160
      medium       0.69      0.70      0.70       410
        high       0.86      0.81      0.84       714

    accuracy                           0.75      1284
   macro avg       0.68      0.71      0.69      1284
weighted avg       0.76      0.75      0.76      1284

Confusion Matrix:
[[ 96  32  32]
 [ 58 289  63]
 [ 37  97 580]]
